# Softmax算子教程

## 概述

本课程将以Softmax算子为例，讲解基于指针的C语言编程的算子开发与优化方法。

我们以昇腾Ascend 950PR/DT (dav-3510 架构)为硬件平台，基于C API接口，按以下步骤实现逐步实现并优化**Softmax**算子：

1. Softmax算子接口分析
2. 基础版Softmax算子实现
3. 优化分析
4. 进阶版Softmax算子实现
5. 优化前后的性能对比

## 环境准备

正式开始学习之前，先要对Jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入Jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
import os

!mkdir -p src
!mkdir -p src_add
import subprocess

result = subprocess.run(
    ['bash', '-l', '-c', 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'],
    capture_output=True, text=True
)
for line in result.stdout.strip().split('\n'):
    line = line.strip()
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value
print("\n🎉 Environment initialization process completed successfully!")

## 1. Softmax算子接口分析

softmax算法的数学公式如下：

$$
\text{Softmax}(x_{i,j}) = \frac{e^{x_{i,j} - \max_i}}{\sum_{j=0}^{n-1} e^{x_{i,j} - \max_i}}
$$

具体实现时，其算法原理如下：

```python
def softmax(src):
    #基于last轴进行rowmax（按行取最大值）处理
    max = np.max(src, axis=-1, keepdims=True)
    sub = src - max
    exp = np.exp(sub)
    #基于last轴进行rowsum（按行求和）处理
    sum = np.sum(exp, axis=-1, keepdims=True)
    dst = exp / sum
    return dst
```

算法是单输入/单输出的，算子接口也只需要设置两个入参，分别用于输入输出即可：

``` c
__global__ __vector__ void softmax_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
```

Device侧需定义核函数所需的数据格式。为简化处理，本文使用固定的数据格式。定义如下常量：

``` c
constexpr uint32_t LOOP_COUNT = 1024;                                   // 单vector核上需要划分成多少个VF
constexpr uint32_t WIDTH = 128;                                         // 输入数据的列数
constexpr uint32_t SINGLE_VF_HEIGHT = 80;                               // 单个VF需要处理的数据行数。
```


## 2. 基础版softmax算子实现

### 1. Device侧（NPU设备侧）功能实现

为方便代码管理，Device侧代码实现放在独立的softmax.h文件中。


- VF函数实现

为了更清晰的展示搬运、计算、同步等各类操作之间的关系，本例采用最直接的方式对输入的每行数据依次完成如下操作：

步骤1: 求每行最大值

步骤2: 对最大值做填充扩展

步骤3: 各元素减去最大值（防止溢出）

步骤4: 指数运算

步骤5: 每行求和

步骤6: 求和结果填充扩展

步骤7: 归一化

In [ ]:
%%writefile src/softmax.h

#include "c_api/asc_simd.h"

namespace Custom {
union DataUnion {
    constexpr __aicore__ DataUnion() : f(0.0f) {}
    constexpr __aicore__ DataUnion(uint32_t val) : i(val) {}
    float f;
    uint32_t i;
};
constexpr DataUnion fp32_min_value(0x00800000u);
}

constexpr uint32_t LOOP_COUNT = 1024;
constexpr uint32_t WIDTH = 128;
constexpr uint32_t SINGLE_VF_HEIGHT = 80;
constexpr uint32_t SINGLE_VF_DATA_LEN = SINGLE_VF_HEIGHT * WIDTH;

__simd_vf__ inline void softmax_vf(__ubuf__ float* dst_ub, __ubuf__ float* src_ub, __ubuf__ float* exp_ub, __ubuf__ float* sum_ub)
{
    constexpr uint16_t one_repeat_cnt = asc_get_vf_len() / sizeof(float);
    uint16_t repeat_times = (WIDTH + one_repeat_cnt - 1) / one_repeat_cnt;
    vector_bool mask_full = asc_create_mask_b32(PAT_ALL);

    vector_bool mask;
    vector_float src_reg;
    vector_float max_reg;
    vector_float exp_reg;
    vector_float sum_reg;
    vector_float div_reg;

    // 第一部分: ReduceMax -> Sub -> Exp
    for (uint16_t i = 0; i < SINGLE_VF_HEIGHT; i++) {
        uint32_t count1 = WIDTH;
        asc_duplicate_scalar(max_reg, Custom::fp32_min_value.f, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count1);
            asc_loadalign(src_reg, src_ub + i * WIDTH + j * one_repeat_cnt);
            asc_max(max_reg, max_reg, src_reg, mask);
        }
        asc_reduce_max(max_reg, max_reg, mask_full);
        asc_duplicate(max_reg, max_reg, mask_full);
        uint32_t count2 = WIDTH;

        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count2);
            asc_loadalign(src_reg, src_ub + i * WIDTH + j * one_repeat_cnt);
            asc_sub(exp_reg, src_reg, max_reg, mask);
            asc_exp(exp_reg, exp_reg, mask);
            asc_storealign(exp_ub + i * WIDTH + j * one_repeat_cnt, exp_reg, mask);
        }
    }

    // 同步：前置步骤中写入exp_ub的操作完成后才能启动后续步骤。
    asc_mem_bar(VST_VLD);

    // 第二部分: ReduceSum -> Div
    for (uint16_t i = 0; i < SINGLE_VF_HEIGHT; i++) {
        uint32_t count3 = WIDTH;
        asc_duplicate_scalar(sum_reg, 0.0, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count3);
            asc_loadalign(src_reg, exp_ub + i * WIDTH + j * one_repeat_cnt);
            asc_add(sum_reg, sum_reg, src_reg, mask);
        }
        asc_reduce_sum(sum_reg, sum_reg, mask_full);
        asc_duplicate(sum_reg, sum_reg, mask_full);
        uint32_t count4 = WIDTH;
        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count4);
            asc_loadalign(max_reg, exp_ub + i * WIDTH + j * one_repeat_cnt);
            asc_div(div_reg, max_reg, sum_reg, mask);
            asc_storealign(dst_ub + i * WIDTH + j * one_repeat_cnt, div_reg, mask);
        }
    }
}


- Device侧入口函数：核函数实现

核函数是Device侧提供的，供Host侧调用的入口函数。它接收Host侧提供的__gm__内存指针、Tiling等数据，负责完成如下工作：

a. 结合当前核的ID号，推算出当前核需要处理的数据段。

b. 申请__ub__内存，将本核需要处理的数据搬运到__ub__中。

c. 调用VF函数完成计算。

d. 将计算结果搬出到__gm__内存中本核对应的地址段中。


In [ ]:
%%writefile -a src/softmax.h

// ======================= 核函数 =========================
__global__ __vector__ void softmax_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
{
    asc_init();

    // 申请UB内存
    __ubuf__ float src_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float dst_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float exp_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float sum_ub[SINGLE_VF_DATA_LEN];

    __gm__ float* x_gm = reinterpret_cast<__gm__ float*>(x);
    __gm__ float* y_gm = reinterpret_cast<__gm__ float*>(y);
    uint8_t mutex_id = 1;

    for (uint32_t i = 0; i < LOOP_COUNT; i++) {
        // 锁定UB资源，确保搬入完成后再执行计算操作。
        asc_lock(PIPE_MTE2, mutex_id);
        asc_set_copy_pad_val(float(0.0));
        for (uint32_t j = 0; j < SINGLE_VF_HEIGHT; j++) {
            asc_copy_gm2ub_align(&src_ub[j * WIDTH], &x_gm[i * SINGLE_VF_DATA_LEN + j * WIDTH], 1, static_cast<uint16_t>(WIDTH * sizeof(float)), 0, 0, true, 0, 0, 0);
        }
        asc_unlock(PIPE_MTE2, mutex_id);

        // 数据搬入操作完成后才能启动计算
        asc_lock(PIPE_V, mutex_id);
        // 调用VF函数执行计算
        softmax_vf(dst_ub, src_ub, exp_ub, sum_ub);
        asc_unlock(PIPE_V, mutex_id);

        // 计算完成后才能启动数据搬出
        asc_lock(PIPE_MTE3, mutex_id);
        for (uint32_t j = 0; j < SINGLE_VF_HEIGHT; j++) {
            asc_copy_ub2gm_align(&y_gm[i * SINGLE_VF_DATA_LEN + j * WIDTH], &dst_ub[j * WIDTH], 1, static_cast<uint16_t>(WIDTH * sizeof(float)), 0, 0, 0);
        }
        asc_unlock(PIPE_MTE3, mutex_id);
    }
}


### 2. Host侧功能实现

Host侧代码是整个算子的入口，运行在CPU上。它负责完成如下功能：

a. 申请Host侧内存空间，并准备输入数据。

b. 准备__gm__内存空间，并将需要计算的数据，拷贝到__gm__内存中。

c. 调用Device侧核函数完成计算。

d. 将计算结果拷贝到Host侧内存中。

e. 校验核函数输出的结果是否符合预期。

为方便代码管理，Host侧代码实现放在独立的softmax.asc中实现。

- 输入数据准备

本段代码在Host侧执行，用于生成指定长度和类型的随机数据，作为softmax算子的输入。

In [ ]:
%%writefile src/softmax.asc

#include <iostream>
#include <vector>
#include <iterator>
#include <random>
#include "acl/acl.h"
#include "softmax.h"

// ======================= 构造随机数据 =========================
int64_t random_key()
{
    static std::random_device rd;
    static std::mt19937_64 gen(rd());
    static std::uniform_int_distribution<int64_t> dist(0, std::numeric_limits<int64_t>::max());
    return dist(gen);
}

float random_value()
{
    static std::random_device rd;
    static std::mt19937_64 gen(rd());
    static std::uniform_real_distribution<double> dist(0.0, 1.0); // 模板随机 value [0,1]
    return static_cast<float>(dist(gen));
}

void generate_test_data(std::vector<int64_t>& keys, std::vector<float>& values, uint32_t num, uint32_t value_dim)
{
    keys.resize(num);
    values.resize(num * value_dim);

    for (uint32_t i = 0; i < num; ++i) {
        keys[i] = random_key();
        for (uint32_t j = 0; j < value_dim; ++j) {
            values[i * value_dim + j] = random_value();
        }
    }
}


- 生成golden数据

本段代码在Host侧执行。根据输入数据，生成softmax的预期结果，将用于校验softmax算子的输出结果是否正确。

In [ ]:
%%writefile -a src/softmax.asc

// ======================= 构造golden数据 =========================
std::vector<float> softmax_reference(std::vector<float>& src, uint32_t height, uint32_t width)
{
    // 支持 float 和 half 数据类型。half 类型先 cast 为 float 进行计算。
    std::vector<float> dst(src.size());
    for (uint32_t row = 0; row < height; row++) {
        uint32_t offset = row * width;
        // 找最大值
        float max_val = static_cast<float>(src[offset]);
        for (uint32_t col = 1; col < width; col++) {
            max_val = std::max(max_val, static_cast<float>(src[offset + col]));
        }
        // 计算 exp(x - max) 并求和
        float sum_val = 0.0f;
        for (uint32_t col = 0; col < width; col++) {
            float tmp = std::exp(static_cast<float>(src[offset + col]) - max_val);
            dst[offset + col] = static_cast<float>(tmp);
            sum_val += tmp;
        }
        // 归一化
        for (uint32_t col = 0; col < width; col++) {
            dst[offset + col] = static_cast<float>(static_cast<float>(dst[offset + col]) / sum_val);
        }
    }
    return dst;
}


- 结果校验

本段代码在Host侧执行。用于校验softmax算子实际输出的结果和Host侧softmax_reference函数生成golden数据是否相同（float数据类型的误差许可范围一般控制在万分之一）。

In [ ]:
%%writefile -a src/softmax.asc

// ======================= 校验输出结果 =========================
uint32_t verify_vesult(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_func = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    std::cout << "Output size" << ": " << output.size() << std::endl;
    std::cout << "Golden size" << ": " << golden.size() << std::endl;
    print_func(output, "Output");
    print_func(golden, "Golden");
    constexpr float epsilon = 1e-4f;
    bool passed = true;
    uint32_t errCount = 0;
    for (size_t i = 0; i < golden.size(); i++) {
        if (std::abs(static_cast<float>(output[i]) - static_cast<float>(golden[i])) > epsilon) {
            passed = false;
            errCount++;
        }
    }
    if (passed) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed! errCount:" << errCount << std::endl;
        return 1;
    }
}


- Host侧入口函数：main函数实现

In [ ]:
%%writefile -a src/softmax.asc

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t dim = 1;
    constexpr uint32_t height = SINGLE_VF_HEIGHT * LOOP_COUNT;
    constexpr uint32_t width = WIDTH;
    constexpr uint32_t totalLen = height * width;

    std::vector<int64_t> keys;
    std::vector<float> src(totalLen);
    generate_test_data(keys, src, height, width);
    size_t inputSize = totalLen * sizeof(float);
    uint8_t* src_device = nullptr;
    uint8_t* dst_device = nullptr;
    
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    
    aclrtMalloc((void**)&src_device, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&dst_device, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    
    aclrtMemcpy(src_device, inputSize, src.data(), inputSize, ACL_MEMCPY_HOST_TO_DEVICE);
    
    softmax_custom<<<dim, 0, stream>>>(src_device, dst_device);
    aclrtSynchronizeStream(stream);
    
    std::vector<float> output(totalLen);
    aclrtMemcpy(output.data(), inputSize, dst_device, inputSize, ACL_MEMCPY_DEVICE_TO_HOST);
    
    std::vector<float> golden = softmax_reference(src, height, width);
    uint32_t result = verify_vesult(output, golden);
    
    aclrtFree(src_device);
    aclrtFree(dst_device);
    
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return result;
}


### 3. CMake编译配置

一个完整的算子应用程序需要通过编译生成可执行文件。对于本例中的softmax算子，我们需要配置CMake以支持Ascend C编译工具链，并指定目标架构为dav-3510（Ascend 950PR/Ascend 950DT）。

注意，动态链接库m包含Host侧生成golden数据时所需的数学计算类的方法。

In [ ]:
%%writefile src/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510")

find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    softmax.asc
)

set(SOFTMAX_COMMON_LIBS  m)
target_link_libraries(demo PRIVATE ${SOFTMAX_COMMON_LIBS})

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

## 3. 优化分析

观察上节中的算子实现，可发现在VF函数中存在如下问题：
- vf内部逐一遍历每一行，没有启用双发能力。
- asc_sub和asc_exp分别计算，没有使用融合指令。
- 核函数内src_ub和dst_ub使用同一个资源锁MTE2/MTE3无法并行。

所以，可通过以下手段提升算子性能：
- 使用asc_exp_sub替代asc_sub和asc_exp，减少指令数。
- VF中展开外层循环，提升双发能力，减少每次遍历的循环次数。
- src_ub和dst_ub使用不同资源锁，提升MTE2/MTE3的并行能力。

## 4. 进阶版Softmax算子实现

优化前后Host侧代码不变。

优化后的softmax.h文件实现如下：

In [ ]:
%%writefile src/softmax.h

#include "c_api/asc_simd.h"

namespace Custom {
union DataUnion {
    constexpr __aicore__ DataUnion() : f(0.0f) {}
    constexpr __aicore__ DataUnion(uint32_t val) : i(val) {}
    float f;
    uint32_t i;
};
constexpr DataUnion fp32MinValue(0x00800000u);
}

constexpr uint32_t LOOP_COUNT = 1024;
constexpr uint32_t WIDTH = 128;
constexpr uint32_t SINGLE_VF_HEIGHT = 80;
constexpr uint32_t SINGLE_VF_DATA_LEN = SINGLE_VF_HEIGHT * WIDTH;

template <typename T>
__simd_vf__ inline void softmax_vf(__ubuf__ T* dstAddr, __ubuf__ T* srcAddr, __ubuf__ float* expAddr, __ubuf__ float* sumAddr)
{
    constexpr uint16_t one_repeat_cnt = asc_get_vf_len() / sizeof(float);
    uint16_t repeat_times = (WIDTH + one_repeat_cnt - 1) / one_repeat_cnt;
    vector_bool mask_full = asc_create_mask_b32(PAT_ALL);

    vector_float srcReg;
    vector_float maxReg;
    vector_float expReg;
    vector_float sumReg;
    vector_float divReg;

    vector_float srcReg1;
    vector_float maxReg1;
    vector_float expReg1;
    vector_float sumReg1;
    vector_float divReg1;

    uint16_t halfA = SINGLE_VF_HEIGHT >> 1;

    // 第一部分: ReduceMax -> Sub -> Exp
    for (uint16_t i = 0; i < halfA; i++) {
        asc_duplicate_scalar(maxReg, Custom::fp32MinValue.f, mask_full);
        asc_duplicate_scalar(maxReg1, Custom::fp32MinValue.f, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(srcReg, srcAddr + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(srcReg1, srcAddr + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);
            asc_max(maxReg, maxReg, srcReg, mask_full);
            asc_max(maxReg1, maxReg1, srcReg1, mask_full);
        }
        asc_reduce_max(maxReg, maxReg, mask_full);
        asc_reduce_max(maxReg1, maxReg1, mask_full);
        asc_duplicate(maxReg, maxReg, mask_full);
        asc_duplicate(maxReg1, maxReg1, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(srcReg, srcAddr + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(srcReg1, srcAddr + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);

            asc_exp_sub(expReg, srcReg, maxReg, mask_full);
            asc_exp_sub(expReg1, srcReg1, maxReg1, mask_full);
            asc_storealign(expAddr + i * WIDTH + j * one_repeat_cnt, expReg, mask_full);
            asc_storealign(expAddr + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH, expReg1, mask_full);
        }
    }

    // 同步：前置步骤中写入exp_ub的操作完成后才能启动后续步骤。
    asc_mem_bar(VST_VLD);

    // 第二部分: ReduceSum -> Div
    for (uint16_t i = 0; i < halfA; i++) {
        asc_duplicate_scalar(sumReg, 0.0, mask_full);
        asc_duplicate_scalar(sumReg1, 0.0, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(srcReg, expAddr + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(srcReg1, expAddr + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);
            asc_add(sumReg, sumReg, srcReg, mask_full);
            asc_add(sumReg1, sumReg1, srcReg1, mask_full);
        }
        asc_reduce_sum(sumReg, sumReg, mask_full);
        asc_reduce_sum(sumReg1, sumReg1, mask_full);
        asc_duplicate(sumReg, sumReg, mask_full);
        asc_duplicate(sumReg1, sumReg1, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(maxReg, expAddr + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(maxReg1, expAddr + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);
            asc_div(divReg, maxReg, sumReg, mask_full);
            asc_div(divReg1, maxReg1, sumReg1, mask_full);
            asc_storealign(dstAddr + i * WIDTH + j * one_repeat_cnt, divReg, mask_full);
            asc_storealign(dstAddr + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH, divReg1, mask_full);
        }
    }
}

// ======================= 核函数 =========================
__global__ __vector__ void softmax_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
{
    asc_init();

    // 申请UB内存
    __ubuf__ float src_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float dst_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float exp_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float sum_ub[SINGLE_VF_DATA_LEN];

    __gm__ float* x_gm = reinterpret_cast<__gm__ float*>(x);
    __gm__ float* y_gm = reinterpret_cast<__gm__ float*>(y);

    uint8_t mutex_id_src = 1;   // src_ub对应的资源锁
    uint8_t mutex_id_dst = 2;   // dst_ub对应的资源锁

    for (uint32_t i = 0; i < LOOP_COUNT; i++) {
        // 搬入操作需要独占src_ub
        asc_lock(PIPE_MTE2, mutex_id_src);
        asc_set_copy_pad_val(float(0.0));
        for (uint32_t j = 0; j < SINGLE_VF_HEIGHT; j++) {
            asc_copy_gm2ub_align(&src_ub[j * WIDTH], &x_gm[i * SINGLE_VF_DATA_LEN + j * WIDTH], 1, static_cast<uint16_t>(WIDTH * sizeof(float)), 0, 0, true, 0, 0, 0);
        }
        asc_unlock(PIPE_MTE2, mutex_id_src);

        // 计算操作需要独占src_ub和dst_ub
        asc_lock(PIPE_V, mutex_id_src);
        asc_lock(PIPE_V, mutex_id_dst);
    
        // 调用VF函数执行计算
        softmax_vf(dst_ub, src_ub, exp_ub, sum_ub);

        // 计算完成，释放资源
        asc_unlock(PIPE_V, mutex_id_src);
        asc_unlock(PIPE_V, mutex_id_dst);

        // 搬出操作需要独占dst_ub
        asc_lock(PIPE_MTE3, mutex_id_dst);
        for (uint32_t j = 0; j < SINGLE_VF_HEIGHT; j++) {
            asc_copy_ub2gm_align(&y_gm[i * SINGLE_VF_DATA_LEN + j * WIDTH], &dst_ub[j * WIDTH], 1, static_cast<uint16_t>(WIDTH * sizeof(float)), 0, 0, 0);
        }
        asc_unlock(PIPE_MTE3, mutex_id_dst);
    }
}


## 5. 优化前后的性能对比

通过流水图对比可以看出，基础版数据搬入、计算、搬出三个过程依次进行。PIPE_MTE2、PIPE_V、PIPE_MTE3三个关键流水中均存在明显空闲时间。

**图1** 基础版流水图

<img src="./images/04.03_softmax/softmax_base_timeline.png" title="基础版流水图" style="zoom:100%;" />

**图2** 进阶版流水图

<img src="./images/04.03_softmax/softmax_improved_timeline.png" title="进阶版流水图" style="zoom:100%;" />


以文中1024 * 80 * 128的数据量为例，关键性能数据如下（单位：微妙）：
|   | vector核总时间 | vector核计算类操作时间 | MTE2搬运操作时间 | MTE3搬运操作时间 |
| :----- | :------- | :------- | :------- | :------- |
| 基础版 | 3851.64  | 728.493 | 2032.885 | 1068.679 |
| 进阶版 | 3330.36  | 645.23  | 2017.652 | 1053.355 |
| 优化比例 | 13.53% | 11.43% | 0.75% | 1.43% |

表中的vector核总时间指所有操作完成时间核启动时间之间的差值。

vector核计算类操作时间、MTE2搬运操作时间、MTE3搬运操作时间指各类操作的有效执行时间的总和，不包括对应流水上的空闲时间。

其中，vector核计算类操作时间的优化主要来自以下两个优化点：
- 使用asc_exp_sub替代asc_sub和asc_exp，减少指令数。
- VF中展开外层循环，提升双发能力，减少每次遍历的循环次数。

vector核总时间的优化是在vector核计算类优化的基础上叠加了如下优化点的收益：
- src_ub和dst_ub使用不同资源锁，提升MTE2/MTE3的并行能力。


## 课后实践

请使用上文中的优化方法，编写一个加法算子，完成一维数组的加法运算。

用户需完成如下3个文件：

1、编写add.asc：在该文件中完成Device侧核函数和VF函数的实现。

In [ ]:
%%writefile src_add/add.asc

2、编写add.h：在该文件中完成Host侧的功能。如测试数据构造、结果校验、核函数调用等。

In [ ]:
%%writefile src_add/add.h

3、编写CMakeLists.txt：在该文件中完成CMake配置。

In [ ]:
%%writefile src_add/CMakeLists.txt

执行如下脚本可验证精度是否符合预期：

In [ ]:
!cd src_add && mkdir -p build && cd build && \
cmake .. && \
make && \
./demo

执行如下命令可查看参考答案：

In [ ]:
!cat answer/04.03/add/add.asc

In [ ]:
!cat answer/04.03/add/add.h

In [ ]:
!cat answer/04.03/add/CMakeLists.txt